# Dataset and PandasLabeledDataProvider tutorial

This notebook demonstrates how to build annotated time series with `PandasLabeledDataProvider`, combine them into `Dataset`, and select bisegments for NoReset experiments.

In [ ]:
import pandas as pd

from pysatl_cpd.core.data_providers.dataset import (
    Annotation,
    Dataset,
    PandasLabeledDataProvider,
)


In [ ]:
ts_one = pd.DataFrame(
    {
        "value": [1.0, 1.2, 1.1, 4.0, 3.9, 8.0, 8.1],
        "aux": [10, 11, 12, 20, 21, 30, 31],
        "segments": [0, 0, 0, 1, 1, 2, 2],
    }
)

segment_info_one = pd.DataFrame(
    {
        "start": [0, 3, 5],
        "end": [2, 4, 6],
        "label": ["stable", "middle", "shifted"],
    }
)

provider_one = PandasLabeledDataProvider(
    dataset=ts_one,
    segment_info=segment_info_one,
    annotation=Annotation(path="ts_one.csv", scenario="A", version="v1"),
    name="series_one",
)

ts_two = pd.DataFrame(
    {
        "value": [0.5, 0.4, 2.5, 2.7],
        "aux": [7, 8, 9, 10],
        "segments": [0, 0, 1, 1],
    }
)

segment_info_two = pd.DataFrame(
    {
        "start": [0, 2],
        "end": [1, 3],
        "label": ["baseline", "changed"],
    }
)

provider_two = PandasLabeledDataProvider(
    dataset=ts_two,
    segment_info=segment_info_two,
    annotation=Annotation(path="ts_two.csv", scenario="B", version="v1"),
    name="series_two",
)

dataset = Dataset([provider_one, provider_two])
dataset

In [ ]:
# 1) Change points are inferred from the `segments` column.
provider_one.change_point


In [ ]:
# 2) Select a subset of features while keeping the internal segmentation.
provider_one_value_only = provider_one.select_columns(["value"])
list(provider_one_value_only)[:3]


In [ ]:
# 3) Filter full dataset by annotation.
scenario_a = dataset.filter_by_annotation(lambda ann: ann.scenario == "A")
len(scenario_a.timeserieses)


In [ ]:
# 4) Select bisegments for NoReset mode.
# Keep only pairs where the next segment starts from index >= 3.
bisegments = dataset.select_bisegments_by_filter(lambda pair: pair[1].start >= 3)
len(bisegments), [b.name for b in bisegments]


In [ ]:
# 5) Inspect one resulting bisegment.
example_bisegment = bisegments[0]
example_bisegment.dataset, example_bisegment.segment_info
